# Predicting House Prices Using Machine Learning (Ames Housing)

**Course:** D7054E Data Science Programming  
**Group:** 38  
**Members:** Avijit Saha, Seyed Morteza Salehi

---

This notebook implements an end-to-end pipeline for the Kaggle *House Prices: Advanced Regression Techniques* dataset:

1. Data retrieval (CSV loading)
2. EDA (plots + missing values)
3. Hypotheses checks
4. Cleaning + preprocessing
5. Feature engineering (beyond baseline)
6. Model selection + cross-validation
7. Results, plots, and interpretation

> **Student-style note:** Comments are written like we are explaining our thinking step-by-step (for ourselves and for the examiner).


## 0) Imports + Global Settings

We keep imports in one place so it's easy to see what libraries are used.
We also set a random seed for reproducibility.


In [ ]:

# Core
from pathlib import Path

import numpy as np
import pandas as pd

# Plots
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_squared_error, r2_score

sns.set(style="whitegrid")
RANDOM_STATE = 42


## 1) Data Retrieval

According to the project specification, we must show how we retrieve the data.
Here we load `train.csv` and `test.csv` downloaded from Kaggle.

**Expected folder structure:**

```
project/
  data/
    train.csv
    test.csv
  images/
  House_Prices_D7054E_Group38.ipynb
```


In [ ]:

# Paths
ROOT_DIR = Path('.').resolve()
DATA_DIR = ROOT_DIR / 'data'
IMAGES_DIR = ROOT_DIR / 'images'
IMAGES_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'

# Small helper so errors are more friendly

def load_csv(path: Path) -> pd.DataFrame:
    """Load a CSV file into a DataFrame."""
    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}. "
            "Please put train.csv and test.csv inside the data/ folder."
        )
    return pd.read_csv(path)

train_df = load_csv(TRAIN_PATH)
test_df = load_csv(TEST_PATH)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
train_df.head()


## 2) Quick Sanity Checks

We do simple checks so we don't waste time later (e.g., missing target column).


In [ ]:

TARGET_COL = 'SalePrice'

if TARGET_COL not in train_df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in training data!")

print('Target column exists ✅')
print('Missing target values:', train_df[TARGET_COL].isna().sum())


## 3) Exploratory Data Analysis (EDA)

Here we explore:
- distribution of `SalePrice` (usually right-skewed)
- missing values
- correlations for numeric features

> **Student-style note:** EDA is basically us trying to understand what the data looks like before ML.


In [ ]:

# SalePrice distribution (raw)
plt.figure(figsize=(7, 4))
sns.histplot(train_df[TARGET_COL], kde=True, color='steelblue')
plt.title('SalePrice distribution (raw)')
plt.xlabel(TARGET_COL)
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'saleprice_raw.png', dpi=160)
plt.show()

# SalePrice distribution (log1p)
train_df['SalePrice_log'] = np.log1p(train_df[TARGET_COL])

plt.figure(figsize=(7, 4))
sns.histplot(train_df['SalePrice_log'], kde=True, color='darkorange')
plt.title('SalePrice distribution (log1p)')
plt.xlabel('log1p(SalePrice)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'saleprice_log.png', dpi=160)
plt.show()


In [ ]:

# Missing values report (top 20)
missing_count = train_df.isna().sum()
missing_pct = (missing_count / len(train_df)) * 100

missing_report = (
    pd.DataFrame({'missing_count': missing_count, 'missing_pct': missing_pct})
    .sort_values('missing_count', ascending=False)
)

missing_report.head(20)


In [ ]:

# Top correlations with SalePrice (numeric only)
numeric_df = train_df.select_dtypes(include=['number']).copy()
correlations = numeric_df.corr(numeric_only=True)[TARGET_COL].dropna()

corr_top = correlations.reindex(correlations.abs().sort_values(ascending=False).index).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(x=corr_top.values, y=corr_top.index, palette='viridis')
plt.title('Top 15 numeric correlations with SalePrice')
plt.xlabel('Correlation')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'top_correlations.png', dpi=160)
plt.show()

corr_top.to_frame('correlation')


## 4) Hypotheses (Quick Checks)

We check two common hypotheses-related features:
- Size: `GrLivArea`
- Quality: `OverallQual`

We use correlation as a simple check (not claiming causality).


In [ ]:

cols_to_check = ['GrLivArea', 'OverallQual', TARGET_COL]
sub = train_df[cols_to_check].dropna()

corr_size = sub['GrLivArea'].corr(sub[TARGET_COL])
corr_quality = sub['OverallQual'].corr(sub[TARGET_COL])

print(f"Corr(GrLivArea, SalePrice)   = {corr_size:.3f}")
print(f"Corr(OverallQual, SalePrice) = {corr_quality:.3f}")


## 5) Feature Engineering (Beyond Baseline)

To show we went beyond a basic baseline, we create:
- `HouseAge = YrSold - YearBuilt`
- `TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF`

These are easy to justify and explain.


In [ ]:


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add engineered features (returns a copy)."""
    df = df.copy()

    if 'YrSold' in df.columns and 'YearBuilt' in df.columns:
        df['HouseAge'] = (df['YrSold'] - df['YearBuilt']).clip(lower=0)

    sf_cols = ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']
    if all(col in df.columns for col in sf_cols):
        df['TotalSF'] = df[sf_cols].fillna(0).sum(axis=1)

    return df

train_fe = add_engineered_features(train_df)
test_fe = add_engineered_features(test_df)

print('New columns added:', sorted(set(train_fe.columns) - set(train_df.columns)))
train_fe[['HouseAge', 'TotalSF']].head()


## 6) Preprocessing Pipeline

We build a preprocessing pipeline:
- Numeric: median imputation + scaling
- Categorical: most-frequent imputation + one-hot encoding

> Student note: Pipeline helps prevent leakage during CV.


In [ ]:

X = train_fe.drop(columns=[TARGET_COL])
y = np.log1p(train_fe[TARGET_COL].values)

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols)
    ],
    remainder='drop'
)

print('Numeric features:', len(numeric_cols))
print('Categorical features:', len(categorical_cols))


## 7) Models + Cross-Validation Evaluation

We compare multiple models:
- LinearRegression (baseline)
- Ridge, Lasso
- RandomForest
- GradientBoosting

Metrics:
- RMSE
- R²

We evaluate on the **log target**.


In [ ]:


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute RMSE."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=10.0, random_state=RANDOM_STATE),
    'Lasso': Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=5000),
    'RandomForest': RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingRegressor(random_state=RANDOM_STATE)
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', model)
    ])

    y_pred = cross_val_predict(pipe, X, y, cv=cv, n_jobs=-1)

    results.append({
        'model': name,
        'rmse_log': rmse(y, y_pred),
        'r2_log': float(r2_score(y, y_pred))
    })

results_df = pd.DataFrame(results).sort_values(by='rmse_log')
results_df


## 8) Best Model Diagnostics

We pick the best model (lowest RMSE) and create:
- Predicted vs Actual plot
- Residuals plot

These are useful for explaining performance in the oral exam.


In [ ]:

# Pick best model
best_model_name = results_df.iloc[0]['model']
print('Best model based on RMSE:', best_model_name)

best_pipe = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', models[best_model_name])
])

best_pred = cross_val_predict(best_pipe, X, y, cv=cv, n_jobs=-1)

# Predicted vs Actual
plt.figure(figsize=(6, 6))
plt.scatter(y, best_pred, alpha=0.4, color='teal')
min_v = min(y.min(), best_pred.min())
max_v = max(y.max(), best_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], 'k--')
plt.title(f'Predicted vs Actual (log target) - {best_model_name}')
plt.xlabel('Actual (log1p)')
plt.ylabel('Predicted (log1p)')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'predicted_vs_actual.png', dpi=160)
plt.show()

# Residuals
residuals = y - best_pred
plt.figure(figsize=(7, 4))
plt.scatter(best_pred, residuals, alpha=0.4, color='purple')
plt.axhline(0, color='black', linestyle='--')
plt.title(f'Residuals vs Predicted (log target) - {best_model_name}')
plt.xlabel('Predicted (log1p)')
plt.ylabel('Residual (Actual - Predicted)')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'residuals.png', dpi=160)
plt.show()

print('Saved plots in:', IMAGES_DIR)


## 9) Train Final Model + Predict Test (Optional Kaggle Submission)

Optional but nice:
- Fit best model on all training data
- Predict on test
- Save `submission.csv`

We undo the log transform using `expm1`.


In [ ]:

# Fit on full training data
best_pipe.fit(X, y)

X_test = test_fe.copy()

test_pred_log = best_pipe.predict(X_test)
test_pred_price = np.expm1(test_pred_log)

submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': test_pred_price
})

out_path = ROOT_DIR / 'submission.csv'
submission.to_csv(out_path, index=False)

submission.head()


## 10) Ethics Notes (Short Reflection)

- Historical house prices may encode socio-economic inequalities.
- Location-related features can act as proxies for sensitive attributes.
- Real deployment should include monitoring and fairness checks.

(We keep this here so we remember to discuss it in the report and oral exam.)
